In [1]:
!pip install -U transformers accelerate sentencepiece

In [2]:
from transformers import pipeline

pipe = pipeline(
    "text-generation",
    model="TinyLlama/TinyLlama-1.1B-Chat-v1.0"
)

print("Model Loaded Successfully")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.29k [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

Model Loaded Successfully


In [3]:
response = pipe(
    "A patient has fever and headache. Give basic health guidance.",
    max_new_tokens=100
)

print(response[0]["generated_text"])

[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=100) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer LlamaTokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


A patient has fever and headache. Give basic health guidance.


In [4]:
response = pipe(
    """
    <|system|>
    You are a helpful medical assistant.

    <|user|>
    A patient has fever and headache.
    Give:
    1. Possible causes
    2. Basic health guidance
    3. When to seek medical help

    <|assistant|>
    """,
    max_new_tokens=200,
    do_sample=True,
    temperature=0.7
)

print(response[0]["generated_text"])

[transformers] Passing `generation_config` together with generation-related arguments=({'do_sample', 'temperature', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=200) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



    <|system|>
    You are a helpful medical assistant.

    <|user|>
    A patient has fever and headache.
    Give:
    1. Possible causes
    2. Basic health guidance
    3. When to seek medical help

    <|assistant|>
    1. Possible causes:
    - Fever (high body temperature)
    - Headache (significant headache with no other symptoms)
    - Fatigue (sweats, dizziness, or fatigue)

    2. Basic health guidance:
    - Wash hands regularly with soap and water
    - Avoid touching your face, such as rubbing your eyes or nose
    - Stay hydrated by drinking fluids regularly
    - Eat nutritious foods such as fruits and vegetables
    - Get enough sleep

    3. When to seek medical help:
    - If the fever is severe or lasts for more than 24 hours
    - If the fever persists for more than 10 days without improvement
    - If the fever is accompanied by other symptoms such as confusion, disorientation, or seizures
   


In [5]:
!pip install -q gradio

In [6]:
def analyze_symptoms(symptoms):

    prompt = f"""
    <|system|>
    You are MediGuide AI.

    Analyze the symptoms.

    Give:
    1. Possible conditions
    2. Risk level (Low/Medium/High)
    3. Basic health guidance
    4. When to seek medical help

    <|user|>
    {symptoms}

    <|assistant|>
    """

    response = pipe(
        prompt,
        max_new_tokens=250,
        do_sample=True,
        temperature=0.7
    )

    return response[0]["generated_text"]

In [7]:
import gradio as gr

with gr.Blocks() as demo:

    gr.Markdown("# 🩺 MediGuide AI")
    gr.Markdown("Offline Medical Reasoning Assistant")

    symptoms = gr.Textbox(
        label="Describe Your Symptoms",
        lines=4
    )

    output = gr.Textbox(
        label="Analysis",
        lines=15
    )

    btn = gr.Button("Analyze")

    btn.click(
        analyze_symptoms,
        inputs=symptoms,
        outputs=output
    )

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://18776cd0b440d38ae8.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
